In [0]:

# ============================================================
# DIM_PRODUCT — STEP 1: CONFIGURATION
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
import uuid

# ============================================================
# PROJECT CONFIGURATION
# Replace placeholders with your environment-specific values.
# ============================================================

PROJECT_ID = "YOUR_GCP_PROJECT_ID"

# GCS Raw Delta
RAW_PATH = "gs://YOUR_RAW_BUCKET_NAME/products/"

# BigQuery dataset
BQ_DATASET = "Gcp_Curated"

# BigQuery tables
DQ_TABLE = (
    f"{PROJECT_ID}.{BQ_DATASET}.DATA_QUALITY_RESULTS"
)

STAGE_TABLE = (
    f"{PROJECT_ID}.{BQ_DATASET}.DIM_PRODUCT_STAGE"
)

TARGET_TABLE = (
    f"{PROJECT_ID}.{BQ_DATASET}.DIM_PRODUCT"
)

REJECT_TABLE = (
    f"{PROJECT_ID}.{BQ_DATASET}.DATA_QUALITY_REJECTS"
)

# Temporary GCS bucket for the Spark BigQuery connector
BQ_TEMP_BUCKET = "gs://YOUR_BQ_TEMP_BUCKET/"

# Watermark table
WATERMARK_TABLE = "YOUR_CATALOG.YOUR_WATERMARK_TABLE"

# Pipeline metadata
PIPELINE_NAME = "product_curated_pipeline"
SOURCE_SYSTEM = "CloudSQL"

# Generate a unique pipeline run ID
RUN_ID = str(uuid.uuid4())

print("Configuration loaded.")
print("RUN_ID:", RUN_ID)


In [0]:
# ============================================================
# DIM_PRODUCT — STEP 2: READ LAST WATERMARK
# ============================================================

watermark_df = (
    spark.table(WATERMARK_TABLE)
    .filter(F.col("table_name") == "DIM_PRODUCT")
    .filter(F.col("status") == "SUCCESS")
)

lwv = watermark_df.select("lwv").first()["lwv"]

print("DIM_PRODUCT LWV:", lwv)

In [0]:
# ============================================================
# DIM_PRODUCT — STEP 3: RAW DELTA + INCREMENTAL FILTER
# ============================================================

# Read products from Raw Delta
raw_products = (
    spark.read
    .format("delta")
    .load(RAW_PATH)
)

# Apply watermark
product_incremental = (
    raw_products
    .filter(F.col("updated_at") > F.lit(lwv))
)

print("Total raw records:", raw_products.count())
print("Incremental records:", product_incremental.count())

display(product_incremental)

In [0]:
# ============================================================
# DIM_PRODUCT — STEP 4: EXACT DUPLICATE REMOVAL
# ============================================================

before_count = product_incremental.count()

product_incremental = (
    product_incremental
    .dropDuplicates()
)

after_count = product_incremental.count()

print("Records before exact dedup:", before_count)
print("Records after exact dedup :", after_count)
print("Exact duplicates removed   :", before_count - after_count)

In [0]:
# ============================================================
# DIM_PRODUCT — STEP 6: SCD1 LATEST RECORD PER PRODUCT
# ============================================================

# For SCD1, only the latest version of each product_id
# should reach the curated dimension.

product_window = (
    Window
    .partitionBy("product_id")
    .orderBy(
        F.col("updated_at").desc()
    )
)

product_scd1 = (
    product_incremental
    .withColumn(
        "row_num",
        F.row_number().over(product_window)
    )
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

print("Records before SCD1 latest-version selection:",
      product_incremental.count())

print("Records after SCD1 latest-version selection:",
      product_scd1.count())

print(
    "Products reduced:",
    product_incremental.count() - product_scd1.count()
)

display(product_scd1.orderBy("product_id"))

In [0]:
# ============================================================
# DIM_PRODUCT — STEP 5: SCD1 LATEST RECORD
# ============================================================

product_window = (
    Window
    .partitionBy("product_id")
    .orderBy(F.col("updated_at").desc())
)

product_scd1 = (
    product_incremental
    .withColumn("row_num", F.row_number().over(product_window))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

print("Records before SCD1 latest selection:",
      product_incremental.count())

print("Records after SCD1 latest selection:",
      product_scd1.count())

print("Records reduced:",
      product_incremental.count() - product_scd1.count())

display(product_scd1.orderBy("product_id"))

In [0]:
# ============================================================
# DIM_PRODUCT — STEP 6: DEFINE DQ RULES
# ============================================================

from pyspark.sql import functions as F


dq_rules = [

    {
        "dq_rule_id": "PROD_001",
        "dq_rule_name": "Product ID Not Null",
        "dq_rule_type": "COMPLETENESS",
        "column_name": "product_id",

        "condition": F.col("product_id").isNotNull()
    },

    {
        "dq_rule_id": "PROD_002",
        "dq_rule_name": "Product Name Not Null",
        "dq_rule_type": "COMPLETENESS",
        "column_name": "product_name",

        "condition": F.col("product_name").isNotNull()
    },

    {
        "dq_rule_id": "PROD_003",
        "dq_rule_name": "Unit Price Valid",
        "dq_rule_type": "VALIDITY",
        "column_name": "unit_price",

        "condition": (
            F.col("unit_price").isNotNull()
            &
            (F.col("unit_price") >= 0)
        )
    },

    {
        "dq_rule_id": "PROD_004",
        "dq_rule_name": "Stock Quantity Valid",
        "dq_rule_type": "VALIDITY",
        "column_name": "stock_quantity",

        "condition": (
            F.col("stock_quantity").isNotNull()
            &
            (F.col("stock_quantity") >= 0)
        )
    }
]

print("Total DQ rules:", len(dq_rules))

In [0]:
# ============================================================
# DIM_PRODUCT — STEP 7: EXECUTE DQ RULES
# ============================================================

total_records = product_scd1.count()

print("Total records:", total_records)


dq_summary = []

dq_failed_records = []


for rule in dq_rules:

    rule_id = rule["dq_rule_id"]
    rule_name = rule["dq_rule_name"]
    rule_type = rule["dq_rule_type"]
    column_name = rule["column_name"]
    condition = rule["condition"]


    # --------------------------------------------------------
    # Records passing the rule
    # --------------------------------------------------------

    passed_df = product_scd1.filter(condition)

    passed_count = passed_df.count()


    # --------------------------------------------------------
    # Records failing the rule
    # --------------------------------------------------------

    failed_df = product_scd1.filter(~condition)

    failed_count = failed_df.count()


    # --------------------------------------------------------
    # Store DQ summary
    # --------------------------------------------------------

    pass_percentage = round(
        (passed_count / total_records) * 100 if total_records != 0 else 0,
        2
    )

    fail_percentage = round(
        (failed_count / total_records) * 100 if total_records != 0 else 0,
        2
    )

    dq_status = (
        "PASS"
        if failed_count == 0
        else "FAIL"
    )


    dq_summary.append({

        "dq_rule_id": rule_id,

        "dq_rule_name": rule_name,

        "dq_rule_type": rule_type,

        "column_name": column_name,

        "total_records": total_records,

        "passed_records": passed_count,

        "failed_records": failed_count,

        "pass_percentage": pass_percentage,

        "fail_percentage": fail_percentage,

        "dq_status": dq_status
    })


    # --------------------------------------------------------
    # Keep actual failed records
    # --------------------------------------------------------

    if failed_count > 0:

        dq_failed_records.append({

            "dq_rule_id": rule_id,

            "dq_rule_name": rule_name,

            "dq_rule_type": rule_type,

            "column_name": column_name,

            "failed_df": failed_df
        })


print("DQ rules executed:", len(dq_summary))

In [0]:
# ============================================================
# DISPLAY DQ SUMMARY
# ============================================================

dq_summary_df = spark.createDataFrame(dq_summary)

display(
    dq_summary_df
    .orderBy("dq_rule_id")
)

In [0]:
# ============================================================
# DISPLAY DQ SUMMARY
# ============================================================

dq_summary_df = spark.createDataFrame(dq_summary)

display(
    dq_summary_df
    .orderBy("dq_rule_id")
)


# ============================================================
# PREPARE DQ RESULTS FOR BIGQUERY
# ============================================================

dq_results_df = (
    dq_summary_df
    .withColumn("dq_result_sk", F.monotonically_increasing_id())
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("table_name", F.lit("DIM_PRODUCT"))
    .withColumn("check_level", F.lit("RECORD"))
    .withColumn("error_message", F.lit(None).cast("string"))
    .withColumn("source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("pipeline_name", F.lit(PIPELINE_NAME))
    .withColumn("execution_timestamp", F.current_timestamp())
    .withColumn("dw_created_at", F.current_timestamp())
    .withColumn(
        "pass_percentage",
        F.col("pass_percentage").cast("decimal(10,4)")
    )
    .withColumn(
        "fail_percentage",
        F.col("fail_percentage").cast("decimal(10,4)")
    )
    .select(
        "dq_result_sk",
        "run_id",
        "table_name",
        "column_name",
        "dq_rule_id",
        "dq_rule_name",
        "dq_rule_type",
        "check_level",
        "total_records",
        "passed_records",
        "failed_records",
        "pass_percentage",
        "fail_percentage",
        "dq_status",
        "error_message",
        "source_system",
        "pipeline_name",
        "execution_timestamp",
        "dw_created_at"
    )
)


# ============================================================
# WRITE DQ RESULTS TO BIGQUERY
# ============================================================

(
    dq_results_df
    .write
    .format("bigquery")
    .option("temporaryGcsBucket", BQ_TEMP_BUCKET)
    .mode("append")
    .save(DQ_TABLE)
)

print("DQ results successfully written to BigQuery.")

In [0]:
# ============================================================
# COMMON DQ — BUILD REJECTED RECORDS
# ============================================================

rejected_dfs = []

for failed_rule in dq_failed_records:

    rule_id = failed_rule["dq_rule_id"]
    rule_name = failed_rule["dq_rule_name"]
    rule_type = failed_rule["dq_rule_type"]
    failed_df = failed_rule["failed_df"]

    # Keep the original source columns for JSON
    record_columns = failed_df.columns

    rejected_df = (
        failed_df

        .withColumn(
            "reject_id",
            F.monotonically_increasing_id()
        )

        .withColumn(
            "run_id",
            F.lit(RUN_ID)
        )

        .withColumn(
            "table_name",
            F.lit("DIM_PRODUCT")
        )

        .withColumn(
            "dq_rule_id",
            F.lit(rule_id)
        )

        .withColumn(
            "dq_rule_name",
            F.lit(rule_name)
        )

        .withColumn(
            "record_key",
            F.col("product_id").cast("string")
        )

        .withColumn(
            "record_data",
            F.to_json(
                F.struct(
                    *[
                        F.col(c).alias(c)
                        for c in record_columns
                    ]
                )
            )
        )

        .withColumn(
            "failure_reason",
            F.lit(rule_name)
        )

        .withColumn(
            "source_system",
            F.lit(SOURCE_SYSTEM)
        )

        .withColumn(
            "pipeline_name",
            F.lit(PIPELINE_NAME)
        )

        .withColumn(
            "rejected_at",
            F.current_timestamp()
        )

        .select(
            "reject_id",
            "run_id",
            "table_name",
            "dq_rule_id",
            "dq_rule_name",
            "record_key",
            "record_data",
            "failure_reason",
            "source_system",
            "pipeline_name",
            "rejected_at"
        )
    )

    rejected_dfs.append(rejected_df)


# ============================================================
# COMBINE ALL FAILED RECORDS
# ============================================================

if rejected_dfs:

    dq_rejected_df = rejected_dfs[0]

    for df in rejected_dfs[1:]:
        dq_rejected_df = dq_rejected_df.unionByName(df)

else:

    dq_rejected_df = None


# ============================================================
# DISPLAY REJECTED RECORDS
# ============================================================

if dq_rejected_df is not None:

    print(
        "Total rejected records:",
        dq_rejected_df.count()
    )

    display(
        dq_rejected_df
        .orderBy("dq_rule_id", "record_key")
    )

else:

    print("No DQ rejected records.")


# ============================================================
# WRITE REJECTED RECORDS TO BIGQUERY
# ============================================================

if dq_rejected_df is not None:

    (
        dq_rejected_df
        .write
        .format("bigquery")
        .option(
            "temporaryGcsBucket",
            BQ_TEMP_BUCKET
        )
        .mode("append")
        .save(REJECT_TABLE)
    )

    print(
        "Rejected records successfully written to BigQuery."
    )

else:

    print(
        "No rejected records to write to BigQuery."
    )

In [0]:
# ============================================================
# STEP 7 — BUILD CLEAN PRODUCT DATA
#           HANDLE NO-RECORD EDGE CASE
# ============================================================

product_scd1_count = product_scd1.count()

print("Product records before DQ:", product_scd1_count)


# ------------------------------------------------------------
# CASE 1 — NO INCREMENTAL RECORDS
# ------------------------------------------------------------

if product_scd1_count == 0:

    product_clean = product_scd1

    print("No incremental product records found.")
    print("Product records after DQ: 0")


# ------------------------------------------------------------
# CASE 2 — RECORDS EXIST BUT NO DQ REJECTIONS
# ------------------------------------------------------------

elif dq_rejected_df is None:

    product_clean = product_scd1

    print("No DQ rejected records.")
    print(
        "All product records passed DQ."
    )
    print(
        "Product records after DQ:",
        product_clean.count()
    )


# ------------------------------------------------------------
# CASE 3 — RECORDS EXIST AND DQ REJECTIONS EXIST
# ------------------------------------------------------------

else:

    # Get keys of records that failed ANY DQ rule
    rejected_keys = (
        dq_rejected_df
        .select("record_key")
        .distinct()
    )

    # Remove all rejected records
    product_clean = (
        product_scd1
        .join(
            rejected_keys,
            product_scd1.product_id.cast("string")
            == rejected_keys.record_key,
            "left_anti"
        )
    )

    print(
        "Product records after DQ:",
        product_clean.count()
    )


# ------------------------------------------------------------
# DISPLAY CLEAN DATA
# ------------------------------------------------------------

if product_clean.count() > 0:

    display(
        product_clean
        .orderBy("product_id")
    )

else:

    print("No clean product records to process.")

In [0]:
# ============================================================
# STEP 8 — CREATE DIM_PRODUCT_STAGE
# ============================================================

STAGE_TABLE = (
    "gcp-data-engineering-504911."
    "Gcp_Curated.DIM_PRODUCT_STAGE"
)

query = f"""
CREATE OR REPLACE TABLE `{STAGE_TABLE}`
(
    product_id INT64,
    product_name STRING,
    category STRING,
    brand STRING,
    unit_price NUMERIC,
    cost_price NUMERIC,
    weight_grams INT64,
    cocoa_percentage NUMERIC,
    stock_quantity INT64,
    is_active BOOL,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
"""

provider = dbutils.credentials.getServiceCredentialsProvider("bigquery")
provider.refresh(Request())

headers = {
    "Authorization": f"Bearer {provider.token}",
    "X-Goog-User-Project": PROJECT_ID,
    "Content-Type": "application/json"
}

url = (
    f"https://bigquery.googleapis.com/bigquery/v2/"
    f"projects/{PROJECT_ID}/queries"
)

body = {
    "query": query,
    "useLegacySql": False,
    "location": "asia-south2"
}

response = requests.post(
    url,
    headers=headers,
    json=body
)

response.raise_for_status()

print("DIM_PRODUCT_STAGE created.")

In [0]:
# ============================================================
# STEP 9 — WRITE CLEAN DATA TO BIGQUERY STAGE
# ============================================================

(
    product_clean
    .select(
        "product_id",
        "product_name",
        "category",
        "brand",
        "unit_price",
        "cost_price",
        "weight_grams",
        "cocoa_percentage",
        "stock_quantity",
        "is_active",
        "created_at",
        "updated_at"
    )
    .write
    .format("bigquery")
    .option(
        "temporaryGcsBucket",
        BQ_TEMP_BUCKET
    )
    .mode("append")
    .save(STAGE_TABLE)
)

print("Clean product data written to DIM_PRODUCT_STAGE.")

In [0]:
display(
    spark.read
    .format("bigquery")
    .option("table", STAGE_TABLE)
    .load()
    .orderBy("product_id")
)

In [0]:
# ============================================================
# DIM_PRODUCT TARGET SCHEMA
# ============================================================

dim_product_df = (
    spark.read
    .format("bigquery")
    .option("table", TARGET_TABLE)
    .load()
)

dim_product_df.printSchema()

In [0]:
# ============================================================
# STEP 10 — PREPARE CLEAN PRODUCT STAGE
# ============================================================

product_clean_stage = (
    product_clean
    .withColumn(
        "record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("product_name"), F.lit("")),
                F.coalesce(F.col("category"), F.lit("")),
                F.coalesce(F.col("brand"), F.lit("")),
                F.coalesce(
                    F.col("unit_price").cast("string"),
                    F.lit("")
                ),
                F.coalesce(
                    F.col("cost_price").cast("string"),
                    F.lit("")
                ),
                F.coalesce(
                    F.col("weight_grams").cast("string"),
                    F.lit("")
                ),
                F.coalesce(
                    F.col("cocoa_percentage").cast("string"),
                    F.lit("")
                ),
                F.coalesce(
                    F.col("stock_quantity").cast("string"),
                    F.lit("")
                ),
                F.coalesce(
                    F.col("is_active").cast("string"),
                    F.lit("")
                )
            ),
            256
        )
    )
)

print("Clean product records:", product_clean_stage.count())

display(
    product_clean_stage
    .orderBy("product_id")
)

In [0]:
# ============================================================
# STEP 11 — WRITE DIM_PRODUCT_STAGE
# ============================================================

(
    product_clean_stage
    .select(
        "product_id",
        "product_name",
        "category",
        "brand",
        "unit_price",
        "cost_price",
        "weight_grams",
        "cocoa_percentage",
        "stock_quantity",
        "is_active",
        "created_at",
        "updated_at",
        "record_hash"
    )
    .write
    .format("bigquery")
    .option(
        "temporaryGcsBucket",
        BQ_TEMP_BUCKET
    )
    .mode("overwrite")
    .save(STAGE_TABLE)
)

print("DIM_PRODUCT_STAGE written successfully.")

In [0]:
# ============================================================
# STEP 12 — SCD1 MERGE INTO DIM_PRODUCT
# ============================================================

MERGE_QUERY = f"""
MERGE `{TARGET_TABLE}` T
USING `{STAGE_TABLE}` S
ON T.product_id = S.product_id

WHEN MATCHED THEN
    UPDATE SET
        product_name = S.product_name,
        category = S.category,
        brand = S.brand,
        unit_price = S.unit_price,
        cost_price = S.cost_price,
        weight_grams = S.weight_grams,
        cocoa_percentage = S.cocoa_percentage,
        stock_quantity = S.stock_quantity,
        is_active = S.is_active,
        created_at = S.created_at,
        updated_at = S.updated_at,
        record_hash = S.record_hash,
        dw_updated_at = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN
    INSERT (
        product_sk,
        product_id,
        product_name,
        category,
        brand,
        unit_price,
        cost_price,
        weight_grams,
        cocoa_percentage,
        stock_quantity,
        is_active,
        created_at,
        updated_at,
        record_hash,
        dw_created_at,
        dw_updated_at
    )
    VALUES (
        ABS(FARM_FINGERPRINT(
            CAST(S.product_id AS STRING)
        )),
        S.product_id,
        S.product_name,
        S.category,
        S.brand,
        S.unit_price,
        S.cost_price,
        S.weight_grams,
        S.cocoa_percentage,
        S.stock_quantity,
        S.is_active,
        S.created_at,
        S.updated_at,
        S.record_hash,
        CURRENT_TIMESTAMP(),
        CURRENT_TIMESTAMP()
    )
"""

In [0]:
# ============================================================
# STEP 13 — REFRESH BIGQUERY CREDENTIAL
#             AND EXECUTE MERGE
# ============================================================

from google.auth.transport.requests import Request
import requests

# Get Databricks BigQuery service credential
provider = dbutils.credentials.getServiceCredentialsProvider("bigquery")

# Refresh token
provider.refresh(Request())

# Build fresh headers
headers = {
    "Authorization": f"Bearer {provider.token}",
    "X-Goog-User-Project": PROJECT_ID,
    "Content-Type": "application/json"
}

# BigQuery REST API URL
url = (
    f"https://bigquery.googleapis.com/bigquery/v2/"
    f"projects/{PROJECT_ID}/queries"
)

# MERGE request
body = {
    "query": MERGE_QUERY,
    "useLegacySql": False,
    "location": "asia-south2"
}

# Execute MERGE
response = requests.post(
    url,
    headers=headers,
    json=body
)

print("HTTP Status:", response.status_code)

if response.status_code != 200:
    print("BigQuery error:")
    print(response.text)
    response.raise_for_status()

merge_result = response.json()

print("MERGE completed successfully.")
print("Statement type:", merge_result.get("statementType"))
print(
    "DML affected rows:",
    merge_result.get("numDmlAffectedRows")
)

In [0]:
# ============================================================
# STEP 14 — VERIFY DIM_PRODUCT AFTER MERGE
# ============================================================

product_target = (
    spark.read
    .format("bigquery")
    .option("table", TARGET_TABLE)
    .load()
)

print("Total DIM_PRODUCT records:", product_target.count())

display(
    product_target
    .orderBy("product_id")
)

In [0]:
# ============================================================
# VERIFY NO INVALID STOCK RECORDS
# ============================================================

invalid_records = (
    product_target
    .filter(F.col("stock_quantity") < 0)
)

print(
    "Invalid stock records in DIM_PRODUCT:",
    invalid_records.count()
)

display(invalid_records)

In [0]:
# ============================================================
# STEP 15 — CALCULATE NEW LWV
# ============================================================

new_lwv = (
    product_clean_stage
    .agg(
        F.max("updated_at").alias("max_updated_at")
    )
    .first()["max_updated_at"]
)

print("New DIM_PRODUCT LWV:", new_lwv)

In [0]:
# ============================================================
# STEP 16 — UPDATE WATERMARK
# ============================================================

if new_lwv is not None:

    spark.sql(f"""
        UPDATE {WATERMARK_TABLE}
        SET
            lwv = TIMESTAMP('{new_lwv}'),
            status = 'SUCCESS'
        WHERE table_name = 'DIM_PRODUCT'
    """)

    print("DIM_PRODUCT LWV updated successfully.")

else:

    print("No clean records found. LWV was not updated.")

In [0]:
# ============================================================
# STEP 17 — VERIFY WATERMARK
# ============================================================

display(
    spark.table(WATERMARK_TABLE)
    .filter(F.col("table_name") == "DIM_PRODUCT")
)